# 🧪 RAG Pipeline Evaluation — Retrieval Test
**4-Stage evaluation pipeline using the golden dataset**

- Stage 0: BM25 vs Vector (individual retriever comparison)
- Stage 1: After RRF Fusion (combined retrieval)
- Stage 2: After Reranking (NDCG, MRR, Precision)
- Stage 3: After LLM Generation (BERTScore, ROUGE-L, Faithfulness)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 0: Install Dependencies & Imports


In [2]:

!pip install langchain langchain-core langchain-community langchain-nvidia-ai-endpoints langchain-huggingface langchain-chroma chromadb sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 

In [3]:
!pip install rank_bm25

In [4]:
!ls drive/MyDrive/acm/ACM_Hackathon_Project

chroma_db_local      golden_dataset_generator.ipynb  requiment.txt
chroma_db_local.zip  ingestion.ipynb		     retrieval_test.ipynb
docs		     parent_store_local		     retriver.ipynb
extracted_images     parent_store_local.zip	     submission.csv
extracted_images_2   Readme.md


In [5]:

import os
import re
import json
import time
import hashlib
import textwrap
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_classic.storage import LocalFileStore, create_kv_docstore
from langchain_classic.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
from langchain_classic.schema import Document

import getpass



In [6]:
current_dir = os.getcwd()


In [7]:
current_dir = f"{current_dir}/drive/MyDrive/acm/ACM_Hackathon_Project"
current_dir

'/content/drive/MyDrive/acm/ACM_Hackathon_Project'

## Step 1: Load Models, Stores & Golden Dataset


In [8]:
if "NVIDIA_API_KEY" in os.environ:
    del os.environ["NVIDIA_API_KEY"]
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA NIM API Key: ")


# === Embedding Model ===
local_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# === Vector Store ===
persist_dir = f"{current_dir}/chroma_db_local/chroma_db_local"  # Adjust path
vectorstore = Chroma(
    collection_name="med_rag_local",
    embedding_function=local_embeddings,
    persist_directory=persist_dir,
)

# === Parent Store ===
parent_store_dir = f"{current_dir}/parent_store_local/parent_store_local"  # Adjust path
fs = LocalFileStore(parent_store_dir)
store = create_kv_docstore(fs)

# === Reranker ===
bge_reranker = CrossEncoder("BAAI/bge-reranker-large", device="cuda")

# === LLMs ===
llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    api_key=os.environ.get("NVIDIA_API_KEY"),
    temperature=0.0,
    max_tokens=2048,
)

vision_llm = ChatNVIDIA(
    model="meta/llama-3.2-90b-vision-instruct",
    api_key=os.environ.get("NVIDIA_API_KEY"),
    temperature=0.0,
    max_tokens=2048,
)

print("✅ Models loaded")


Enter your NVIDIA NIM API Key: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Models loaded


/tmp/ipykernel_583/3456541696.py:31: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(
/tmp/ipykernel_583/3456541696.py:38: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  vision_llm = ChatNVIDIA(


In [9]:

# === Load Golden Dataset ===
golden_path = f"{current_dir}/docs/golden_dataset.json"
with open(golden_path, "r", encoding="utf-8") as f:
    golden_dataset = json.load(f)

print(f"✅ Loaded {len(golden_dataset)} golden dataset entries")

# Preview
for entry in golden_dataset[:3]:
    print(f"  [{entry['id']}] {entry['question_type']}: {entry['question'][:60]}...")


✅ Loaded 31 golden dataset entries
  [gt_001] simple: What technique developed by Carl Rogers is known as client-c...
  [gt_002] simple: What is the title of the section in the chapter Introduction...
  [gt_003] simple: What is the title of the section that starts on page 173?...


## Step 2: Load Pipeline Functions
Import the same retrieval functions from your retriver notebook.


In [10]:

# === BM25 Retriever ===
# Load all child docs for BM25
all_docs_data = vectorstore.get(include=["documents", "metadatas"])

all_child_docs = []
for i, text in enumerate(all_docs_data["documents"]):
    meta = all_docs_data["metadatas"][i] if i < len(all_docs_data["metadatas"]) else {}
    all_child_docs.append(Document(page_content=text, metadata=meta))

my_bm25_retriever = BM25Retriever.from_documents(all_child_docs, k=5)
print(f"✅ BM25 index built with {len(all_child_docs)} documents")


✅ BM25 index built with 7775 documents


In [11]:

# === HyDE Generator ===
def generate_hyde_vector(query, llm_client, embedding_model):
    prompt = f"""Given the following question, write a hypothetical, detailed textbook
passage that directly answers it.
Question: {query}
Hypothetical Textbook Passage:"""
    response = llm_client.invoke([HumanMessage(content=prompt)])
    return embedding_model.embed_query(response.content)


def get_query_vector(query, embedding_model, llm=None, use_hyde=True):
    if use_hyde and llm:
        return generate_hyde_vector(query, llm, embedding_model)
    else:
        return embedding_model.embed_query(query)


In [23]:

# === Retrieval Functions ===
def retrieve_bm25_only(query, bm25_retriever, top_k=5):
    bm25_retriever.k = top_k
    return bm25_retriever.invoke(query)


def retrieve_vector_only(query_vector, vector_store, top_k=5):
    return vector_store.similarity_search_by_vector(query_vector, k=top_k)

def retrieve_and_fuse(query, hyde_vector, bm25_retriever, vector_store, top_k=5):
    bm25_retriever.k = top_k
    with ThreadPoolExecutor(max_workers=2) as executor:
        bm25_future = executor.submit(bm25_retriever.invoke, query)
        vector_future = executor.submit(vector_store.similarity_search_by_vector, hyde_vector, k=top_k)
        bm25_docs = bm25_future.result()
        vector_docs = vector_future.result()

    fused_scores = {}
    rrf_k = 60
    def score_docs(docs, weight=1.0):
        for rank, doc in enumerate(docs):
            doc_id = doc.page_content
            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": doc, "score": 0.0, "sources": 0}
            fused_scores[doc_id]["score"] += weight / (rank + rrf_k)
            fused_scores[doc_id]["sources"] += 1  # Track how many retrievers found it

    score_docs(bm25_docs, weight=0.4)
    score_docs(vector_docs, weight=1.0)

    reranked = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)

    # ✅ KEY FIX: Only keep docs found by BOTH retrievers, or top-scored ones
    filtered = [
        item for item in reranked
        if item["sources"] >= 2 or item["score"] >= 0.02  # Tune threshold
    ]

    # Fallback: if too few remain, take top-k by score
    if len(filtered) < 3:
        filtered = reranked[:top_k]

    return [item["doc"] for item in filtered[:top_k]], bm25_docs, vector_docs


def rerank_chunks(query, fused_docs, reranker_model, top_n=3, min_score=0.01):
    """Rerank with score threshold to filter out irrelevant chunks."""
    if not fused_docs:
        return []
    pairs = [[query, doc.page_content] for doc in fused_docs]
    scores = reranker_model.predict(pairs)

    # Filter by minimum relevance score, then sort
    scored_docs = [(s, d) for s, d in zip(scores, fused_docs) if s > min_score]
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    kept = [doc for _, doc in scored_docs[:top_n]]
    filtered_out = len(fused_docs) - len(scored_docs)
    if filtered_out > 0:
        print(f"    🔍 Reranker filtered out {filtered_out} low-relevance chunks")
    return kept



def fetch_and_deduplicate_parents(top_child_chunks, doc_store):
    unique_parent_ids = set()
    final_parent_docs = []
    for child in top_child_chunks:
        parent_id = child.metadata.get('doc_id')
        if parent_id and parent_id not in unique_parent_ids:
            unique_parent_ids.add(parent_id)
            parent_bytes = doc_store.mget([parent_id])[0]
            if parent_bytes:
                parent_text = parent_bytes.decode('utf-8') if isinstance(parent_bytes, bytes) else str(parent_bytes)
                final_parent_docs.append({
                    "text": parent_text,
                    "chapter": child.metadata.get("chapter", "Unknown"),
                    "section": child.metadata.get("section", "Unknown"),
                    "section_title": child.metadata.get("section-title", "Unknown"),
                    "page_start": child.metadata.get("page_start", "N/A"),
                    "page_end": child.metadata.get("page_end", "N/A"),
                })
    return final_parent_docs


## Step 3: Evaluation Metrics


In [24]:

# ==========================================
# RETRIEVAL METRICS (Stage 0, 1, 2)
# ==========================================

def compute_recall_at_k(retrieved_docs, golden_sections, k=5):
    """What fraction of relevant sections appear in the retrieved docs?"""
    retrieved_sections = set()
    for doc in retrieved_docs[:k]:
        sec = doc.metadata.get("section", "")
        if sec:
            retrieved_sections.add(sec)
        # Also check parent section (e.g., "6.3" matches "6.3.1")
        parts = sec.split(".")
        if len(parts) >= 2:
            retrieved_sections.add(f"{parts[0]}.{parts[1]}")

    golden_set = set(golden_sections)
    if not golden_set:
        return 1.0  # No ground truth sections = skip

    hits = len(golden_set & retrieved_sections)
    return hits / len(golden_set)


def compute_hit_rate(retrieved_docs, golden_sections, k=5):
    """Did ANY relevant section appear in top-K?"""
    retrieved_sections = set()
    for doc in retrieved_docs[:k]:
        sec = doc.metadata.get("section", "")
        if sec:
            retrieved_sections.add(sec)
            parts = sec.split(".")
            if len(parts) >= 2:
                retrieved_sections.add(f"{parts[0]}.{parts[1]}")

    golden_set = set(golden_sections)
    if not golden_set:
        return 1.0
    return 1.0 if (golden_set & retrieved_sections) else 0.0


def compute_mrr(retrieved_docs, golden_sections, k=5):
    """Mean Reciprocal Rank — how early does the first relevant chunk appear?"""
    golden_set = set(golden_sections)
    if not golden_set:
        return 1.0

    for rank, doc in enumerate(retrieved_docs[:k], 1):
        sec = doc.metadata.get("section", "")
        parts = sec.split(".")
        parent_sec = f"{parts[0]}.{parts[1]}" if len(parts) >= 2 else sec
        if sec in golden_set or parent_sec in golden_set:
            return 1.0 / rank
    return 0.0


def compute_precision_at_k(retrieved_docs, golden_sections, k=3):
    """What fraction of top-K retrieved docs are relevant?"""
    golden_set = set(golden_sections)
    if not golden_set:
        return 1.0

    relevant_count = 0
    for doc in retrieved_docs[:k]:
        sec = doc.metadata.get("section", "")
        parts = sec.split(".")
        parent_sec = f"{parts[0]}.{parts[1]}" if len(parts) >= 2 else sec
        if sec in golden_set or parent_sec in golden_set:
            relevant_count += 1
    return relevant_count / k


def compute_ndcg_at_k(retrieved_docs, golden_sections, k=3):
    """NDCG@K — measures ranking quality. Higher = relevant docs ranked higher."""
    golden_set = set(golden_sections)
    if not golden_set:
        return 1.0

    # Relevance scores: 1 if relevant, 0 if not
    relevances = []
    for doc in retrieved_docs[:k]:
        sec = doc.metadata.get("section", "")
        parts = sec.split(".")
        parent_sec = f"{parts[0]}.{parts[1]}" if len(parts) >= 2 else sec
        rel = 1.0 if (sec in golden_set or parent_sec in golden_set) else 0.0
        relevances.append(rel)

    # DCG
    dcg = sum(rel / np.log2(rank + 2) for rank, rel in enumerate(relevances))

    # Ideal DCG (all relevant docs at the top)
    ideal_rels = sorted(relevances, reverse=True)
    idcg = sum(rel / np.log2(rank + 2) for rank, rel in enumerate(ideal_rels))

    if idcg == 0:
        return 0.0
    return dcg / idcg


In [25]:

# ==========================================
# GENERATION METRICS (Stage 3)
# ==========================================

def compute_rouge_l(prediction, reference):
    # Clean JSON artifacts if present
    if not prediction or not reference:
        return 0.0

    # Strip any JSON wrapper
    try:
        parsed = json.loads(prediction)
        if isinstance(parsed, dict):
            prediction = parsed.get("answer", prediction)
    except (json.JSONDecodeError, TypeError):
        pass

    try:
        parsed = json.loads(reference)
        if isinstance(parsed, dict):
            reference = parsed.get("ground_truth", reference)
    except (json.JSONDecodeError, TypeError):
        pass

    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    m, n = len(pred_tokens), len(ref_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i-1] == ref_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_length = dp[m][n]
    precision = lcs_length / m if m > 0 else 0
    recall = lcs_length / n if n > 0 else 0

    if precision + recall == 0:
        return 0.0
    f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 4)



def compute_bertscore_simple(prediction, reference, embedding_model):
    """Sentence-level BERTScore for better granularity."""
    pred_sents = [s.strip() for s in re.split(r'[.!?]+', prediction) if len(s.strip()) > 10]
    ref_sents = [s.strip() for s in re.split(r'[.!?]+', reference) if len(s.strip()) > 10]

    if not pred_sents or not ref_sents:
        pred_vec = np.array(embedding_model.embed_query(prediction))
        ref_vec = np.array(embedding_model.embed_query(reference))
        return round(float(np.dot(pred_vec, ref_vec) / (np.linalg.norm(pred_vec) * np.linalg.norm(ref_vec))), 4)

    pred_vecs = [np.array(embedding_model.embed_query(s)) for s in pred_sents]
    ref_vecs = [np.array(embedding_model.embed_query(s)) for s in ref_sents]

    # Precision: for each pred sentence, best match in reference
    precision_scores = []
    for pv in pred_vecs:
        sims = [np.dot(pv, rv) / (np.linalg.norm(pv) * np.linalg.norm(rv)) for rv in ref_vecs]
        precision_scores.append(max(sims))

    # Recall: for each ref sentence, best match in prediction
    recall_scores = []
    for rv in ref_vecs:
        sims = [np.dot(pv, rv) / (np.linalg.norm(pv) * np.linalg.norm(rv)) for pv in pred_vecs]
        recall_scores.append(max(sims))

    precision = sum(precision_scores) / len(precision_scores)
    recall = sum(recall_scores) / len(recall_scores)

    if precision + recall == 0:
        return 0.0
    f1 = 2 * precision * recall / (precision + recall)
    return round(float(f1), 4)



def compute_faithfulness(query, answer, context_text, judge_llm):
    """LLM-as-judge: is the answer grounded in the context?"""
    prompt = f"""Evaluate if this answer is FAITHFUL to the provided context.
A faithful answer uses ONLY information from the context, with no external knowledge.

Question: {query}
Context: {context_text[:3000]}
Answer: {answer}

Score 1-5:
- 1: Completely hallucinated, no grounding in context
- 2: Mostly hallucinated with some correct info
- 3: Mix of grounded and non-grounded claims
- 4: Mostly grounded with minor additions
- 5: Fully grounded, every claim supported by context

Return JSON: {{"faithfulness": N, "reason": "brief explanation"}}
Respond with ONLY JSON."""

    try:
        response = judge_llm.invoke([HumanMessage(content=prompt)])
        result = json.loads(response.content)
        return result
    except Exception:
        return {"faithfulness": 3, "reason": "evaluation failed"}


def compute_completeness(query, answer, ground_truth, judge_llm):
    """LLM-as-judge: does the answer cover all aspects of the ground truth?"""
    prompt = f"""Compare the model's answer against the ground truth answer.
How complete is the model's answer?

Question: {query}
Ground Truth: {ground_truth}
Model Answer: {answer}

Score 1-5:
- 1: Completely misses the point
- 2: Covers <25% of the ground truth
- 3: Covers ~50% of the ground truth
- 4: Covers ~75% of the ground truth
- 5: Covers everything in the ground truth (may have extra correct info)

Return JSON: {{"completeness": N, "reason": "brief explanation"}}
Respond with ONLY JSON."""

    try:
        response = judge_llm.invoke([HumanMessage(content=prompt)])
        return json.loads(response.content)
    except Exception:
        return {"completeness": 3, "reason": "evaluation failed"}


def check_citation_accuracy(result_citations, golden_sections, golden_pages):
    """Check if the RAG citations match the golden dataset."""
    if not result_citations:
        return {"section_match": 0.0, "page_match": 0.0}

    cited_sections = set()
    cited_pages = set()
    for c in result_citations:
        sec = c.get("section", "")
        if sec:
            cited_sections.add(sec)
        ps = c.get("page_start")
        pe = c.get("page_end")
        if ps and pe:
            for p in range(int(ps), int(pe) + 1):
                cited_pages.add(p)

    golden_sec_set = set(golden_sections)
    golden_page_set = set(golden_pages)

    sec_match = len(cited_sections & golden_sec_set) / max(len(golden_sec_set), 1)
    page_match = len(cited_pages & golden_page_set) / max(len(golden_page_set), 1) if golden_page_set else 1.0

    return {"section_match": round(sec_match, 4), "page_match": round(page_match, 4)}


## Step 4: Run Full Evaluation Pipeline


In [26]:
USE_HYDE = True
TOP_K_RETRIEVE = 7    # Increased from 5 to get more candidates
TOP_N_RERANK = 3
MIN_RERANK_SCORE = 0.1  # Filter out irrelevant chunks

results = []
answerable_entries = [e for e in golden_dataset if e.get("answerable", True)]
unanswerable_entries = [e for e in golden_dataset if not e.get("answerable", True)]

print(f"📊 Evaluating {len(answerable_entries)} answerable + "
      f"{len(unanswerable_entries)} unanswerable questions")
print(f"   Config: USE_HYDE={USE_HYDE}, TOP_K={TOP_K_RETRIEVE}, "
      f"TOP_N_RERANK={TOP_N_RERANK}, MIN_SCORE={MIN_RERANK_SCORE}")
print("=" * 70)


📊 Evaluating 31 answerable + 0 unanswerable questions
   Config: USE_HYDE=True, TOP_K=7, TOP_N_RERANK=3, MIN_SCORE=0.1


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

print_lock = threading.Lock()

def evaluate_single_query(i, entry):
    query = entry["question"]
    golden_sections = entry.get("relevant_sections", [])
    golden_pages = entry.get("relevant_pages", [])
    ground_truth = entry.get("ground_truth", "")

    eval_result = {"id": entry["id"], "question": query, "type": entry["question_type"]}

    # ── These run SEQUENTIALLY (GPU-bound, not safe to parallelize) ──
    query_vec = get_query_vector(query, local_embeddings, llm, use_hyde=USE_HYDE)
    bm25_docs = retrieve_bm25_only(query, my_bm25_retriever, top_k=TOP_K_RETRIEVE)
    vector_docs = retrieve_vector_only(query_vec, vectorstore, top_k=TOP_K_RETRIEVE)

    eval_result["stage_0"] = {
        "bm25_recall@k": compute_recall_at_k(bm25_docs, golden_sections, TOP_K_RETRIEVE),
        "bm25_hit_rate": compute_hit_rate(bm25_docs, golden_sections, TOP_K_RETRIEVE),
        "vector_recall@k": compute_recall_at_k(vector_docs, golden_sections, TOP_K_RETRIEVE),
        "vector_hit_rate": compute_hit_rate(vector_docs, golden_sections, TOP_K_RETRIEVE),
    }

    fused_docs, _, _ = retrieve_and_fuse(query, query_vec, my_bm25_retriever, vectorstore, TOP_K_RETRIEVE)

    eval_result["stage_1"] = {
        "fused_recall@k": compute_recall_at_k(fused_docs, golden_sections, TOP_K_RETRIEVE),
        "fused_hit_rate": compute_hit_rate(fused_docs, golden_sections, TOP_K_RETRIEVE),
        "context_precision": compute_precision_at_k(fused_docs, golden_sections, TOP_K_RETRIEVE),
    }

    reranked_docs = rerank_chunks(query, fused_docs, bge_reranker, top_n=TOP_N_RERANK, min_score=MIN_RERANK_SCORE)

    eval_result["stage_2"] = {
        "ndcg@k": compute_ndcg_at_k(reranked_docs, golden_sections, TOP_N_RERANK),
        "mrr": compute_mrr(reranked_docs, golden_sections, TOP_N_RERANK),
        "precision@k": compute_precision_at_k(reranked_docs, golden_sections, TOP_N_RERANK),
    }

    parent_docs = fetch_and_deduplicate_parents(reranked_docs, store)
    context_text = "\n\n".join([d["text"][:1500] for d in parent_docs])

    # Build prompt
    formatted_contexts = []
    for ci, doc in enumerate(parent_docs):
        source_tag = f"[Source {ci+1}]"
        meta_line = (f"Chapter: {doc['chapter']} | Section: {doc['section']} "
                     f"| Title: {doc['section_title']} | Pages: {doc['page_start']}-{doc['page_end']}")
        formatted_contexts.append(f"--- {source_tag} ({meta_line}) ---\n{doc['text']}\n")
    combined_context = "\n".join(formatted_contexts)

    json_schema = textwrap.dedent("""\
    {
      "answer": "Detailed answer with inline citations like [Source 1].",
      "citations": [{"source_tag": "[Source 1]", "chapter": "Chapter name", "section": "Section number", "section_name": "Title", "page_start": 1, "page_end": 2}]
    }""")

    system_prompt = f"""You are an expert academic assistant.

STRICT RULES:
1. ONLY use information from the provided textbook context below. DO NOT use your own knowledge.
2. Every factual claim MUST have an inline citation like [Source 1].
3. If a claim cannot be directly supported by the context, DO NOT include it.
4. If the context does not contain enough information, explicitly state so.
5. Prefer quoting or closely paraphrasing the textbook.
6. Return JSON format below.

JSON Schema:
{json_schema}

Textbook Context:
{combined_context}

Question: {query}

Answer (use ONLY the textbook context above, cite every claim):"""

    try:
        response = vision_llm.invoke([HumanMessage(content=[{"type": "text", "text": system_prompt}])])
        raw_answer = response.content
        parsed = json.loads(raw_answer)
        model_answer = parsed.get("answer", raw_answer)
        model_citations = parsed.get("citations", [])
    except Exception:
        model_answer = raw_answer if 'raw_answer' in dir() else ""
        model_citations = []

    # ── These 2 LLM judge calls CAN run in parallel ──
    with ThreadPoolExecutor(max_workers=2) as judge_pool:
        faith_future = judge_pool.submit(compute_faithfulness, query, model_answer, context_text, llm)
        comp_future = judge_pool.submit(compute_completeness, query, model_answer, ground_truth, llm)
        faithfulness_result = faith_future.result()
        completeness_result = comp_future.result()

    rouge_l = compute_rouge_l(model_answer, ground_truth)
    bert_score = compute_bertscore_simple(model_answer, ground_truth, local_embeddings)
    citation_acc = check_citation_accuracy(model_citations, golden_sections, golden_pages)

    eval_result["stage_3"] = {
        "rouge_l": rouge_l,
        "bert_score": bert_score,
        "faithfulness": faithfulness_result.get("faithfulness", 0),
        "faithfulness_reason": faithfulness_result.get("reason", ""),
        "completeness": completeness_result.get("completeness", 0),
        "completeness_reason": completeness_result.get("reason", ""),
        "citation_section_match": citation_acc["section_match"],
        "citation_page_match": citation_acc["page_match"],
    }
    eval_result["model_answer"] = model_answer

    with print_lock:
        print(f"[{i+1}/{len(answerable_entries)}] {query[:50]}... | "
              f"Faith={eval_result['stage_3']['faithfulness']}/5 | "
              f"BERT={bert_score:.3f}")

    return eval_result


# === Run ===
results = []
for i, entry in enumerate(answerable_entries):
    result = evaluate_single_query(i, entry)
    results.append(result)


[1/31] What technique developed by Carl Rogers is known a... | Faith=4/5 | BERT=0.629


In [19]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

print_lock = threading.Lock()

def evaluate_unanswerable(entry, idx, total):
    """Evaluate a single unanswerable question (thread-safe)."""
    query = entry["question"]

    with print_lock:
        print(f"  ⏳ [{idx+1}/{total}] {query[:60]}...")

    eval_result = {"id": entry["id"], "question": query, "type": "unanswerable"}

    query_vec = get_query_vector(query, local_embeddings, llm, use_hyde=USE_HYDE)
    fused_docs, _, _ = retrieve_and_fuse(query, query_vec, my_bm25_retriever, vectorstore, TOP_K_RETRIEVE)
    reranked_docs = rerank_chunks(query, fused_docs, bge_reranker, top_n=TOP_N_RERANK)
    parent_docs = fetch_and_deduplicate_parents(reranked_docs, store)

    formatted_contexts = []
    for ci, doc in enumerate(parent_docs):
        formatted_contexts.append(f"--- [Source {ci+1}] ---\n{doc['text']}\n")
    combined_context = "\n".join(formatted_contexts)

    json_schema = '{"answer": "Answer or state the info is not available.", "citations": []}'

    prompt = f"""You are an expert academic assistant. Answer based strictly on the context.
If the answer is NOT in the context, explicitly state that.
Return JSON: {json_schema}

Context: {combined_context}
Question: {query}
Answer:"""

    try:
        response = vision_llm.invoke([HumanMessage(content=[{"type": "text", "text": prompt}])])
        parsed = json.loads(response.content)
        model_answer = parsed.get("answer", response.content)
    except Exception:
        try:
            model_answer = response.content
        except Exception:
            model_answer = ""

    refusal_keywords = [
        "not contain", "not available", "cannot be answered",
        "does not", "no information", "not enough", "not found",
        "not mentioned", "not provided", "not present",
        "insufficient", "outside the scope"
    ]
    correctly_refused = any(kw in model_answer.lower() for kw in refusal_keywords)

    eval_result["stage_3"] = {
        "correctly_refused": correctly_refused,
        "model_answer": model_answer[:200],
    }
    eval_result["model_answer"] = model_answer

    status = "✅ Refused" if correctly_refused else "❌ HALLUCINATED"
    with print_lock:
        print(f"  {status} [{idx+1}/{total}] {query[:50]}...")

    return eval_result


# === Run in parallel ===
print(f"\n{'='*70}")
print(f"🚫 Evaluating {len(unanswerable_entries)} UNANSWERABLE questions (parallel)")
print("=" * 70)

MAX_WORKERS = 3  # 3 concurrent LLM calls (adjust based on API rate limits)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(evaluate_unanswerable, entry, i, len(unanswerable_entries)): entry
        for i, entry in enumerate(unanswerable_entries)
    }

    for future in as_completed(futures):
        result = future.result()
        results.append(result)

correct = sum(1 for r in results if r["type"] == "unanswerable" and r["stage_3"]["correctly_refused"])
total = len(unanswerable_entries)
print(f"\n✅ Done! Correct refusals: {correct}/{total} ({correct/max(total,1)*100:.0f}%)")



🚫 Evaluating 0 UNANSWERABLE questions (parallel)

✅ Done! Correct refusals: 0/0 (0%)


## Step 5: Aggregate Results & Report


In [21]:
# === Aggregate Scores ===
answerable_results = [r for r in results if r["type"] != "unanswerable"]
unanswerable_results = [r for r in results if r["type"] == "unanswerable"]

def avg(values):
    return round(sum(values) / max(len(values), 1), 4)

def pct(val):
    return round(val * 100, 1)

print("=" * 70)
print("📊 EVALUATION REPORT")
print("=" * 70)

if answerable_results:
    s0 = [r["stage_0"] for r in answerable_results if "stage_0" in r]
    s1 = [r["stage_1"] for r in answerable_results if "stage_1" in r]
    s2 = [r["stage_2"] for r in answerable_results if "stage_2" in r]
    s3 = [r["stage_3"] for r in answerable_results if "stage_3" in r]

    # ── Stage 0 ──
    print(f"\n── Stage 0: Individual Retrievers ──")
    print(f"  BM25   Recall@{TOP_K_RETRIEVE}:   {pct(avg([x['bm25_recall@k'] for x in s0]))}%")
    print(f"  BM25   Hit Rate:     {pct(avg([x['bm25_hit_rate'] for x in s0]))}%")
    print(f"  Vector Recall@{TOP_K_RETRIEVE}:   {pct(avg([x['vector_recall@k'] for x in s0]))}%")
    print(f"  Vector Hit Rate:     {pct(avg([x['vector_hit_rate'] for x in s0]))}%")

    # ── Stage 1 ──
    print(f"\n── Stage 1: After RRF Fusion ──")
    print(f"  Fused Recall@{TOP_K_RETRIEVE}:    {pct(avg([x['fused_recall@k'] for x in s1]))}%")
    print(f"  Fused Hit Rate:      {pct(avg([x['fused_hit_rate'] for x in s1]))}%")
    print(f"  Context Precision:   {pct(avg([x['context_precision'] for x in s1]))}%")

    # ── Stage 2 ──
    print(f"\n── Stage 2: After Reranking ──")
    print(f"  NDCG@{TOP_N_RERANK}:          {pct(avg([x['ndcg@k'] for x in s2]))}%")
    print(f"  MRR:              {pct(avg([x['mrr'] for x in s2]))}%")
    print(f"  Precision@{TOP_N_RERANK}:      {pct(avg([x['precision@k'] for x in s2]))}%")

    # ── Stage 3 (simplified) ──
    answer_quality = round(avg([(s["bert_score"] * 0.7 + s["rouge_l"] * 0.3) * 100 for s in s3]), 1)
    trustworthiness = round(avg([s["faithfulness"] / 5.0 * 100 for s in s3]), 1)
    coverage = round(avg([s["completeness"] / 5.0 * 100 for s in s3]), 1)
    overall_gen = round(answer_quality * 0.4 + trustworthiness * 0.3 + coverage * 0.3, 1)

    print(f"\n── Stage 3: Generation Quality ──")
    print(f"  Answer Quality:      {answer_quality}%")
    print(f"  Trustworthiness:     {trustworthiness}%")
    print(f"  Coverage:            {coverage}%")
    print(f"  ⭐ Generation Score:  {overall_gen}%")

# ── Hallucination Resistance ──
if unanswerable_results:
    correct_refusals = sum(1 for r in unanswerable_results if r["stage_3"]["correctly_refused"])
    total_unans = len(unanswerable_results)
    print(f"\n── Hallucination Resistance ──")
    print(f"  Correct Refusals:    {correct_refusals}/{total_unans} ({correct_refusals/total_unans*100:.1f}%)")

# ── Latency ──
if answerable_results and "timings" in answerable_results[0]:
    timings_all = [r["timings"] for r in answerable_results if "timings" in r]
    print(f"\n── Latency (avg per query) ──")
    for key in timings_all[0]:
        vals = [t.get(key, 0) for t in timings_all]
        print(f"  {key:20s}: {avg(vals):.3f}s")
    total_times = [sum(t.values()) for t in timings_all]
    print(f"  {'TOTAL':20s}: {avg(total_times):.3f}s")

print(f"\n{'=' * 70}")


📊 EVALUATION REPORT

── Stage 0: Individual Retrievers ──
  BM25   Recall@7:   59.7%
  BM25   Hit Rate:     67.7%
  Vector Recall@7:   79.0%
  Vector Hit Rate:     87.1%

── Stage 1: After RRF Fusion ──
  Fused Recall@7:    79.0%
  Fused Hit Rate:      90.3%
  Context Precision:   31.8%

── Stage 2: After Reranking ──
  NDCG@3:          66.6%
  MRR:              65.0%
  Precision@3:      43.0%

── Stage 3: Generation Quality ──
  Answer Quality:      54.0%
  Trustworthiness:     79.4%
  Coverage:            66.5%
  ⭐ Generation Score:  65.4%



## Step 6: Save Detailed Results


In [22]:

# Save full results
output_path = f"{current_dir}/docs/evaluation_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

print(f"✅ Detailed results saved to {output_path}")

# Save summary
summary = {
    "config": {"use_hyde": USE_HYDE, "top_k": TOP_K_RETRIEVE, "top_n_rerank": TOP_N_RERANK},
    "stage_0": {
        "bm25_recall": avg([r["stage_0"]["bm25_recall@k"] for r in answerable_results if "stage_0" in r]),
        "vector_recall": avg([r["stage_0"]["vector_recall@k"] for r in answerable_results if "stage_0" in r]),
    },
    "stage_1": {
        "fused_recall": avg([r["stage_1"]["fused_recall@k"] for r in answerable_results if "stage_1" in r]),
        "context_precision": avg([r["stage_1"]["context_precision"] for r in answerable_results if "stage_1" in r]),
    },
    "stage_2": {
        "ndcg": avg([r["stage_2"]["ndcg@k"] for r in answerable_results if "stage_2" in r]),
        "mrr": avg([r["stage_2"]["mrr"] for r in answerable_results if "stage_2" in r]),
    },
    "stage_3": {
        "rouge_l": avg([r["stage_3"]["rouge_l"] for r in answerable_results if "stage_3" in r]),
        "bert_score": avg([r["stage_3"]["bert_score"] for r in answerable_results if "stage_3" in r]),
        "faithfulness": avg([r["stage_3"]["faithfulness"] for r in answerable_results if "stage_3" in r]),
        "completeness": avg([r["stage_3"]["completeness"] for r in answerable_results if "stage_3" in r]),
    },
    "hallucination_resistance": (
        sum(1 for r in unanswerable_results if r["stage_3"]["correctly_refused"]) / max(len(unanswerable_results), 1)
        if unanswerable_results else None
    ),
}

summary_path = f"{current_dir}/docs/evaluation_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"✅ Summary saved to {summary_path}")
print(json.dumps(summary, indent=2))


✅ Detailed results saved to /content/drive/MyDrive/acm/ACM_Hackathon_Project/docs/evaluation_results.json
✅ Summary saved to /content/drive/MyDrive/acm/ACM_Hackathon_Project/docs/evaluation_summary.json
{
  "config": {
    "use_hyde": true,
    "top_k": 7,
    "top_n_rerank": 3
  },
  "stage_0": {
    "bm25_recall": 0.5968,
    "vector_recall": 0.7903
  },
  "stage_1": {
    "fused_recall": 0.7903,
    "context_precision": 0.318
  },
  "stage_2": {
    "ndcg": 0.6656,
    "mrr": 0.6505
  },
  "stage_3": {
    "rouge_l": 0.1889,
    "bert_score": 0.6901,
    "faithfulness": 3.9677,
    "completeness": 3.3226
  },
  "hallucination_resistance": null
}
